# Paquetes

In [ ]:
using PhysicalConstants
using PhysicalConstants.CODATA2022: ħ, m_e, c_0
using Unitful
using Plots
using CSV
using DataFrames
using Measures

# Funciones

In [ ]:
function Kronig_Penney(E, ω, b, V₀; m_eff = 0.067, normalizado = false, compacto = false)
    
    a = ω + b
    
    # estas dos variables las añado para que el coseno de k₁ y k₂ no haga overflow
    mₑc² = ustrip(uconvert(u"eV", m_e*c_0^2))                 # en EV
    ħc = ustrip(uconvert(u"eV*nm", ħ*c_0))                    # en eV nm

    k₁ = @. sqrt(2 * m_eff * mₑc² * E) / ħc                   # en nm⁻¹
    k₂ = @. sqrt(Complex(2 * m_eff * mₑc² * (E - V₀))) / ħc   # en nm⁻¹
    
    # Valor de cos(ka), con esto vemos las bandas imponiendo que f este entre -1 y +1
    f = @. cos(k₁*ω)*cos(k₂*b) - ((k₁^2 + k₂^2) / (2 * k₁ * k₂)) * sin(k₁*ω)*sin(k₂*b)
    
    banda = real.(f)
    
    k_eff, E_eff = Calcular_k(E, banda, a, normalizado, compacto)
    
    return banda, k_eff, E_eff
end



function Calcular_k(E, band, a, normalizar, compactar)
    
    Es = []
    ks = []
    counter = -0.5 * !compactar
    
    for i in 2:length(band)
        if ((abs(band[i]) < 1.0) ⊻ (abs(band[i-1]) < 1.0))
            counter += 0.5 * !compactar
        end
        if (abs(band[i]) <= 1.0)
            push!(Es, E[i])
            if normalizar
                push!(ks, (1 - 2 * !es_par(counter)) * (acos(band[i])/π) + es_entero(counter) * (counter + !es_par(counter)))           # en unidades de π/a
            else
                push!(ks, (1 - 2 * !es_par(counter)) * (acos(band[i])/a) + es_entero(counter) * (counter + !es_par(counter) ) * π / a)  # k en nm⁻¹
            end
        end
    end
    
    return ks, Es
end

function es_par(x)
    return x % 2 == 0
end

function es_entero(x)
    return x % 1 == 0
end

function compute_v(E, k; primer_orden = true, limit = 7e6)
    v = zeros(length(E))
    eV_to_J = 1.602176634e-19
    nm_inv_to_m_inv = 1e9
    for i in 2:length(E)-!primer_orden
        Δk = (k[i] - k[i-1]) * nm_inv_to_m_inv
        if primer_orden
            ΔE = (E[i] - E[i-1]) * eV_to_J
            v[i] = ΔE / Δk / ustrip(ħ)
            if v[i] > limit
                v[i] = NaN
            end
        else
            ΔE = (E[i+1] - E[i-1]) * eV_to_J
            v[i] = ΔE / 2Δk / ustrip(ħ)
            if v[i] > limit
                v[i] = NaN
            end
        end
    end
    return v
end

function n_to_k_fit(n, N, a; normalizado = true)
    k = @. 2 / N * (0.5014 * n + 0.3036) * (π / a)^!normalizado
    return k 
end


function leer_archivo_csv(carpeta, archivo)
    ruta = joinpath(@__DIR__, "..", carpeta, archivo)
    ruta = normpath(ruta)
    println(ruta)
    data = CSV.read(ruta, DataFrame, header = 4)
    return data
end

function renormalizar(x,lim)
    mask = x .< lim
    maximo = maximum(x[mask])
    x_norm = copy(x)
    x_norm = @. x/maximo
    return x_norm, maximo
end

# Datos

In [ ]:
Energy = 0:0.00005:1.0
Barrier_Potential = 0.3
Barrier_width = 15.0
Well_width = 3.0
banda, k_bandas, E_bandas = Kronig_Penney(Energy,Well_width,Barrier_width,Barrier_Potential, m_eff = 0.023, normalizado = false, compacto = false)

# Gráficos

In [ ]:
p1 = plot(Energy, banda, ylims = [-2.0,2.0], legend = false, xlabel = "E (eV)", ylabel = "f(E)", size = (400,400), left_margin = 3mm, bottom_margin = 3mm, grid = false)
hline!([1.0, -1.0])
p2 = plot(k_bandas,E_bandas, legend = false, xlabel = "k (nm⁻¹)", ylabel = "E (eV)", size = (400,400), grid = false)

p_12 = plot(p1,p2, layout = (1,2), size = (800,400))
savefig(p_12, "graficas_banda")

In [ ]:
display(p2)

In [ ]:
prueba = leer_archivo_csv("Pruebas", "100_pozos_juande_params_preciso.txt")
v_g = compute_v(E_bandas,k_bandas, primer_orden = false)
velocidad = plot(k_bandas[1:end-1],v_g[1:end-1], xlabel = "k (nm⁻¹)", ylabel = "v (m/s)", label = "Teórico")

ocurrencias = prueba.Occurrence
E_sim = prueba[:,2]
a = Barrier_width + Well_width
N = 100
k_sim_fit = n_to_k_fit(ocurrencias, N, a, normalizado = false)
k_sim = (prueba.Occurrence .- 0.5) ./ N .* π ./ a
v_sim = compute_v(E_sim, k_sim, primer_orden = false)
p2 = plot(k_bandas,E_bandas, label = "teorica", xlim = [0,0.3])
plot!(p2, k_sim, prueba[:,2], label = "simulacion fit")
plot!(p2, k_sim_fit, prueba[:,2], label = "simulacion k teorica")
plot!(velocidad, k_sim_fit[1:end-1], v_sim[1:end-1], label = "simulación fit")
plot!(velocidad, k_sim[1:end-1], v_sim[1:end-1], label = "simulacion k teorica")
display(p2)
display(velocidad)


In [ ]:
prueba = leer_archivo_csv("Pruebas", "100_pozos_juande_params_preciso.txt")
v_g = compute_v(E_bandas,k_bandas)
v_g_norm, maxi = renormalizar(v_g, 7e6)
velocidad = plot(k_bandas,v_g_norm, xlabel = "k (nm⁻¹)", ylabel = "v (m/s)", label = "Teórico")

ocurrencias = prueba.Occurrence
E_sim = prueba[:,2]
a = Barrier_width + Well_width
N = 100
k_sim_fit = n_to_k_fit(ocurrencias, N, a, normalizado = false)
k_sim = (prueba.Occurrence .- 0.5) ./ N .* π ./ a
v_sim = compute_v(E_sim, k_sim) ./ maxi
plot!(velocidad, k_sim_fit, v_sim, label = "simulación fit")
# plot!(velocidad, k_sim, v_sim, label = "simulacion k teorica")
plot!(k_sim_fit, prueba[:,2], label = "simulacion fit")
# plot!(k_sim, prueba[:,2], label = "simulacion k teorica")
plot!(k_bandas,E_bandas, label = "teorica")
